# NAT Workflow
Create and run NAT workflows.

NAT workflows are defined in YAML configuration files that specify your agent's capabilities, which LLM to use, and which tools it can access. This config-driven approach means you can test different models or swap tools by changing a few lines—no code refactoring required.
Build a simple climate Q&A assistant, test it locally, deploy it as a REST API, and preview the final application you'll build.

In [1]:
%%capture
#install Nemo Agent Toolkit and langchain dependency
!pip install nvidia-nat
!pip install "nvidia-nat[langchain]"

In [2]:
# load env variables 
from dotenv import load_dotenv
import os
   
# Load environment variables from .env file
load_dotenv()
   
# Verify the key loaded
print("API key loaded:", "Yes" if os.getenv('NVIDIA_API_KEY') else "No")

API key loaded: Yes


In [3]:
%%writefile config.yml

llms:
  climate_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    base_url: $NVIDIA_BASE_URL
    api_key: $NVIDIA_API_KEY
    temperature: 0.7
    max_tokens: 2048

workflow:
  _type: chat_completion
  llm_name: climate_llm
  system_prompt: |
    You are a knowledgeable climate science assistant. You help users understand 
    climate data, weather patterns, and global temperature trends. Be accurate, 
    informative, and cite scientific consensus when appropriate.

Overwriting config.yml


In [4]:
# Simple question about climate basics
!nat run \
  --config_file config.yml \
  --input "What is the difference between weather and climate?"

2026-04-16 23:35:26 - INFO     - matplotlib.font_manager:1639 - generated new fontManager
2026-04-16 23:35:33 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yml'
2026-04-16 23:35:34 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Numb

In [5]:
# Question about global temperature trends
!nat run \
  --config_file config.yml \
  --input "How much has global average \
  temperature increased since pre-industrial times?"

2026-04-16 23:35:59 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yml'
2026-04-16 23:36:00 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number

In [6]:
# A data-specific question that would benefit from real data analysis
!nat run \
  --config_file config.yml \
  --input "What were the exact temperature anomalies \
  for the top 5 warmest countries in 2023?"

2026-04-16 23:36:22 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yml'
2026-04-16 23:36:22 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.
/usr/local/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:171: UserWarning: http://jupyter-api-proxy.internal.dlai/rev-proxy/nvidia does not end in /v1, you may have inference and listing issues. This check will be deprecated in the next release. Please ensure /v1 is appended to the provided URL
  warnings.warn(

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number

## Deploy as an API
Now make agent accessible through a REST API endpoint using `nat serve`.

In production, you can run the following line in a terminal:
```bash
nat serve --config_file config.yml
```

In [7]:
# use subprocess since you are running in a notebook
import subprocess
import time

# Clean up old processes if you ran notebook before
!pkill -f "nat serve" 2>/dev/null || true
time.sleep(2)

# Start NAT with explicit IPv4 host
nat_process = subprocess.Popen(
    ["nat", "serve", "--config_file", "config.yml", "--host", "127.0.0.1"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait and let it print to console naturally
time.sleep(25)

In [8]:
import requests
import json

# Test the API endpoint
response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    headers={"Content-Type": "application/json"},
    json={
        "messages": [
            {
                "role": "user",
                "content": "What causes El Niño and how does it affect global weather?"
            }
        ],
        "stream": False
    }
)

# Parse and display the response
if response.status_code == 200:
    result = response.json()
    print(result["choices"][0]["message"]["content"])
else:
    print(f"Error: {response.status_code}")
    print(response.text)

El Niño is a complex weather phenomenon that occurs when there's an abnormal warming of the ocean water temperatures in the eastern Pacific, near the equator. This warming of the ocean water temperatures is caused by a weakening of the trade winds, which normally blow from east to west along the equator.

During a typical year, the trade winds push warm water from the western Pacific towards the eastern Pacific. However, when the trade winds weaken or even reverse direction, the warm water flows back towards the eastern Pacific, causing an increase in ocean temperatures. This warming of the ocean water temperatures is what we call El Niño.

The effects of El Niño on global weather are widespread and significant. Some of the most notable impacts include:

1. **Heavy rainfall and flooding**: El Niño tends to bring heavy rainfall to the eastern Pacific, including countries such as Peru and Ecuador. This can lead to severe flooding and landslides.
2. **Drought in Australia and Indonesia**:

## Clean Up

In [10]:
ui_manager.stop()
nat_process.terminate()
nat_process.wait()
print("✅ Server stopped")

🛑 Stopping UI server...
✅ UI server stopped
✅ Server stopped
